# 03 — Walk-Forward Backtesting

**Pipeline stage 3 of 5**

## Objective
Replace the single 80/20 split used in the original MVP with rolling-window,
walk-forward validation, evaluated **separately per tier** (7–14, 30, 60–90
day horizons). This is the notebook that makes the README's quant claims
true rather than aspirational — its output (`backtest_results.parquet`) is
what `quant/backtesting.py` in the backend should be built from.

## Why walk-forward, not a single split
A single 80/20 split tells you how the model does on one fixed future window.
It doesn't tell you whether performance holds up consistently, or degrades
in specific seasons/years. Walk-forward re-trains on an expanding window and
tests on each subsequent slice — much closer to how the model will actually
be used in production (retrained periodically, always predicting forward).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
import warnings

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "data" / "processed"

TIER_HORIZONS = {
    "tier_7_14":  {"horizon_steps": 2, "label": "7-14 days"},
    "tier_30":    {"horizon_steps": 4, "label": "30 days"},
    "tier_60_90": {"horizon_steps": 8, "label": "60-90 days"},
}
# horizon_steps is expressed in *observations ahead*, consistent with the
# observation-based lag design from notebook 02.

## 1. Walk-forward split generator

Expanding-window: each fold trains on everything up to a cut point and tests on the next `horizon_steps` observations. This is run per crop×market group, not globally, since series lengths and gap structure differ (see notebook 01's coverage report).

In [ ]:
def walk_forward_folds(n_rows: int, horizon_steps: int, min_train: int = 12, step: int = 1):
    """Yields (train_end_idx, test_end_idx) index pairs for expanding-window walk-forward CV."""
    folds = []
    train_end = min_train
    while train_end + horizon_steps <= n_rows:
        folds.append((train_end, train_end + horizon_steps))
        train_end += step
    return folds

# Sanity check on fold count for a mid-sized series
print("Example folds (n_rows=40, horizon=4):", walk_forward_folds(40, 4)[:5], "... total:", len(walk_forward_folds(40, 4)))

## 2. Backtest runner (XGBoost)

Same model family as production (`XGBRegressor`), run per tier per crop×market. `try/except` around each group is deliberate: some crop×market pairs will be too short for a given tier's `min_train` requirement, and that should be recorded as a coverage gap, not crash the whole backtest.

In [ ]:
from xgboost import XGBRegressor

def backtest_group(group: pd.DataFrame, feature_cols: list, horizon_steps: int, min_train: int = 12):
    group = group.sort_values("date").reset_index(drop=True)
    folds = walk_forward_folds(len(group), horizon_steps, min_train=min_train)
    if not folds:
        return None

    fold_metrics = []
    for train_end, test_end in folds:
        train = group.iloc[:train_end]
        test = group.iloc[train_end:test_end]
        if len(test) == 0 or train["price"].std() == 0:
            continue

        model = XGBRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0,
        )
        model.fit(train[feature_cols], train["price"])
        preds = model.predict(test[feature_cols])

        fold_metrics.append({
            "mae": mean_absolute_error(test["price"], preds),
            "mape": mean_absolute_percentage_error(test["price"], preds),
            "r2": r2_score(test["price"], preds) if len(test) > 1 else np.nan,
            "n_test": len(test),
        })

    if not fold_metrics:
        return None
    m = pd.DataFrame(fold_metrics)
    return {
        "n_folds": len(m),
        "mae_mean": m["mae"].mean(), "mae_std": m["mae"].std(),
        "mape_mean": m["mape"].mean(), "mape_std": m["mape"].std(),
        "r2_mean": m["r2"].mean(),
    }

## 3. Run backtest across all tiers and crop×market pairs

This is the expensive cell — expect it to take a while on the full dataset. Results are cached to parquet so notebooks 04/05 don't need to re-run it.

In [ ]:
results = []
for tier, cfg in TIER_HORIZONS.items():
    feat = pd.read_parquet(PROCESSED_DIR / f"features_{tier}.parquet")
    feature_cols = [c for c in feat.columns if c not in ("price", "date")]

    # crop_enc/market_enc identify the group; recover readable labels for reporting
    for (crop_enc, market_enc), group in feat.groupby(["crop_enc", "market_enc"]):
        metrics = backtest_group(group, feature_cols, cfg["horizon_steps"])
        if metrics is None:
            continue
        metrics.update({"tier": tier, "tier_label": cfg["label"], "crop_enc": crop_enc, "market_enc": market_enc})
        results.append(metrics)

backtest_results = pd.DataFrame(results)
backtest_results.to_parquet(PROCESSED_DIR / "backtest_results.parquet", index=False)
backtest_results.shape

## 4. Aggregate results per tier

This is the table that should replace the README's single blended MAE/MAPE/R² figures — reported per tier, as promised.

In [ ]:
tier_summary = (
    backtest_results.groupby("tier_label")
    .agg(
        n_crop_market_pairs=("mae_mean", "count"),
        mae_mean_ugx=("mae_mean", "mean"),
        mape_mean_pct=("mape_mean", lambda s: s.mean() * 100),
        r2_mean=("r2_mean", "mean"),
    )
    .reindex(["7-14 days", "30 days", "60-90 days"])
)
tier_summary

**Expected pattern:** MAE/MAPE should increase and R² should decrease moving from the 7-14 day tier to the 60-90 day tier — if it doesn't, that's a signal to re-check the lag/horizon configuration in notebook 02 before trusting the numbers.

## Output

- `data/processed/backtest_results.parquet` — per crop×market, per tier fold-level metrics

**Next:** `04_prediction_intervals_risk.ipynb`